# NiN classification — raster-based

Rasterizes all input layers to the photic zone grid, classifies each pixel to a NiN type,
adds a 1-pixel AOI padding, vectorizes with GDAL, then subtracts land using STRtree.

In [69]:
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rasterio.warp
from rasterio.features import rasterize as rio_rasterize
from scipy.ndimage import binary_dilation, distance_transform_edt
from shapely.geometry import box as shapely_box, MultiPolygon, Polygon
from shapely.ops import unary_union
from shapely.strtree import STRtree
from osgeo import gdal, ogr, osr

import mnk.substrat as subkart

CRS = "EPSG:25833"


## Load source data

In [84]:
gdf_torrfall = subkart.sources.sea_map_torrfall(["More_og_Romsdal"])
gdf_brakkvann = gpd.read_file(
    "gs://niva-geodata/MarintNaturKart/aux/brackish_water_polygons_buff160.gpkg"
).to_crs(CRS)


## Grid and Lys layer

Use the photic zone raster as the reference grid (50 m, EPSG:25833).

In [71]:
photic_path = "nisjedata-fotisk-sone_moere-og-romsdal_2026_25833.tif"

with rasterio.open(photic_path) as src:
    transform = src.transform
    out_shape  = (src.height, src.width)
    lys = src.read(1)  # 1=Eufotisk, 0=Afotisk, 255=nodata
    mr_box = gpd.GeoDataFrame(geometry=[shapely_box(*src.bounds)], crs=CRS)

print(f'Grid: {out_shape}')
print(f'Eufotisk: {np.sum(lys==1):,}  Afotisk: {np.sum(lys==0):,}  NoData: {np.sum(lys==255):,}')


Grid: (3673, 4349)
Eufotisk: 782,290  Afotisk: 2,067,062  NoData: 13,124,525


## Rasterize Salinitet

Rasterize the dedicated brakkvann polygons as Brakk (1); everything else → Salt (0).

In [72]:
salinitet = rio_rasterize(
    [(geom, 1) for geom in gdf_brakkvann.geometry],
    out_shape=out_shape, transform=transform,
    fill=0, dtype=np.uint8, all_touched=True,
)
print(f'Salt: {np.sum(salinitet==0):,}  Brakk: {np.sum(salinitet==1):,}')


Salt: 15,967,468  Brakk: 6,409


## Rasterize Tørrfall

Innenfor (1) = within tørrfall zone; Utenfor (0) = outside.

In [73]:
torrfall = rio_rasterize(
    [(geom, 1) for geom in gdf_torrfall.geometry],
    out_shape=out_shape, transform=transform,
    fill=0, dtype=np.uint8, all_touched=False,
)
print(f'Innenfor: {np.sum(torrfall==1):,}  Utenfor: {np.sum(torrfall==0):,}')


Innenfor: 60,340  Utenfor: 15,913,537


## Rasterize Substrat

Load the vector substrat parquet, clip to Møre og Romsdal, then rasterize.
løsbunn/blanding → 0 (Løst), fastbunn → 1 (Fast).

In [74]:
gdf_substrat = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat_norge_2026_25833.geo.parquet"
)
# Filter to polygons intersecting Møre og Romsdal — keep full geometries so large
# cross-boundary polygons are not clipped (rasterize handles the extent itself)
gdf_substrat = gdf_substrat[gdf_substrat.intersects(mr_box.geometry.iloc[0])]

BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=Løst, 1=Fast
substrat = rio_rasterize(
    [(geom, BUNN_TO_INT[bt]) for geom, bt in zip(gdf_substrat.geometry, gdf_substrat['BunnType'])
     if bt in BUNN_TO_INT],
    out_shape=out_shape, transform=transform,
    fill=255, dtype=np.uint8, all_touched=True,
)
print(f'Løst: {np.sum(substrat==0):,}  Fast: {np.sum(substrat==1):,}  NoData: {np.sum(substrat==255):,}')


Løst: 2,039,208  Fast: 1,282,354  NoData: 12,652,315


## NiN pixel classification

Encode 4 binary dimensions into a 4-bit key:

```
key = lys*8 + salinitet*4 + substrat*2 + torrfall
```

LUT maps key → NiN integer (1–9); undefined combinations → 255.

In [ ]:
NODATA = 255

# NiN integer (1–12) → (NiN_code, NiN_name)
# Each of the 12 (Lys, Salinitet, Substrat, Tørrfall) combinations gets a unique int
NIN_INT_MAP = {
     1: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Eufotisk Brakk Løst  Innenfor
     2: ("NA-MA04", "Fjærebelte-sedimentbunn"),          # Eufotisk Salt  Løst  Innenfor
     3: ("NA-MC01", "Fast brakkvanns-fjærebeltebunn"),   # Eufotisk Brakk Fast  Innenfor
     4: ("NA-MA01", "Fast saltvanns-fjærebeltebunn"),    # Eufotisk Salt  Fast  Innenfor
     5: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Eufotisk Brakk Løst  Utenfor
     6: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Afotisk  Brakk Løst  Utenfor
     7: ("NA-MA05", "Eufotisk saltvanns-sedimentbunn"),  # Eufotisk Salt  Løst  Utenfor
     8: ("NA-MA06", "Afotisk saltvanns-sedimentbunn"),   # Afotisk  Salt  Løst  Utenfor
     9: ("NA-MC02", "Fast brakkvannsbunn"),              # Eufotisk Brakk Fast  Utenfor
    10: ("NA-MC02", "Fast brakkvannsbunn"),              # Afotisk  Brakk Fast  Utenfor
    11: ("NA-MA02", "Eufotisk fast saltvannsbunn"),      # Eufotisk Salt  Fast  Utenfor
    12: ("NA-MA03", "Afotisk fast saltvannsbunn"),       # Afotisk  Salt  Fast  Utenfor
}

# LUT[key] → nin_int (1–12);  Afotisk+Innenfor combos (keys 1,3,5,7) → 255
# key = lys*8 + salinitet*4 + substrat*2 + torrfall
LUT = np.array([8, 255, 12, 255, 6, 255, 10, 255, 7, 2, 11, 4, 5, 1, 9, 3], dtype=np.uint8)

valid = (lys != NODATA) & (salinitet != NODATA) & (substrat != NODATA)
key = np.full(out_shape, NODATA, dtype=np.uint8)
key[valid] = (
    lys[valid].astype(np.uint16) * 8
    + salinitet[valid].astype(np.uint16) * 4
    + substrat[valid].astype(np.uint16) * 2
    + torrfall[valid].astype(np.uint16)
).astype(np.uint8)

nin_raster = np.full(out_shape, NODATA, dtype=np.uint8)
nin_raster[valid] = LUT[key[valid]]

for code_int, (nin_code, nin_name) in NIN_INT_MAP.items():
    print(f'{code_int:2d}  {nin_code}  {nin_name}: {np.sum(nin_raster==code_int):,}')


 1  NA-MC04  Brakkvanns-sedimentbunn: 326
 2  NA-MA04  Brakkvanns-sedimentbunn: 26,768
 3  NA-MC01  Fast brakkvanns-fjærebeltebunn: 100
 4  NA-MA01  Fast saltvanns-fjærebeltebunn: 26,921
 5  NA-MC04  Brakkvanns-sedimentbunn: 1,423
 6  NA-MC04  Brakkvanns-sedimentbunn: 1,929
 7  NA-MA05  Eufotisk saltvanns-sedimentbunn: 291,276
 8  NA-MA06  Afotisk saltvanns-sedimentbunn: 1,424,893
 9  NA-MC02  Fast brakkvannsbunn: 279
10  NA-MC02  Fast brakkvannsbunn: 701
11  NA-MA02  Eufotisk fast saltvannsbunn: 429,915
12  NA-MA03  Afotisk fast saltvannsbunn: 592,186


## 1-pixel AOI padding

Dilate the valid AOI mask by one pixel and fill boundary pixels
with the nearest valid NiN value.

In [76]:
valid_mask  = nin_raster != NODATA
boundary    = binary_dilation(valid_mask) & ~valid_mask
_, nn_idx   = distance_transform_edt(~valid_mask, return_indices=True)

nin_padded = nin_raster.copy()
nin_padded[boundary] = nin_raster[nn_idx[0][boundary], nn_idx[1][boundary]]
print(f'Boundary pixels added: {np.sum(boundary):,}')


Boundary pixels added: 98,486


## Save NiN raster

In [77]:
nin_raster_path = "nin_moere-og-romsdal_2026_25833.tif"

with rasterio.open(
    nin_raster_path, 'w', driver='GTiff',
    height=out_shape[0], width=out_shape[1], count=1,
    dtype=np.uint8, crs=CRS, transform=transform,
    nodata=NODATA, compress='deflate', tiled=True,
) as dst:
    dst.write(nin_padded, 1)
print('Saved:', nin_raster_path)


Saved: nin_moere-og-romsdal_2026_25833.tif


## Vectorize with GDAL

In [78]:
nin_vec_path = "nin_moere-og-romsdal_2026_25833.gpkg"
gdal.UseExceptions()

ds   = gdal.Open(nin_raster_path)
band = ds.GetRasterBand(1)
arr  = band.ReadAsArray()

drv_mem  = gdal.GetDriverByName('Mem')
mask_ds  = drv_mem.Create('', ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Byte)
mask_ds.SetGeoTransform(ds.GetGeoTransform())
mask_ds.GetRasterBand(1).WriteArray((arr != NODATA).astype(np.uint8))

srs = osr.SpatialReference()
srs.ImportFromEPSG(25833)

Path(nin_vec_path).unlink(missing_ok=True)
out_ds    = ogr.GetDriverByName('GPKG').CreateDataSource(nin_vec_path)
out_layer = out_ds.CreateLayer('nin', srs=srs, geom_type=ogr.wkbPolygon)
out_layer.CreateField(ogr.FieldDefn('nin_int', ogr.OFTInteger))

gdal.Polygonize(band, mask_ds.GetRasterBand(1), out_layer, 0, [], callback=gdal.TermProgress_nocb)
out_ds = mask_ds = ds = None
print('Vectorized to:', nin_vec_path)


0...10...20...30...40...50...60...70...80...90...100 - done.
Vectorized to: nin_moere-og-romsdal_2026_25833.gpkg


## Add NiN_code / NiN_name

In [79]:
gdf_nin_vec = gpd.read_file(nin_vec_path).explode(index_parts=False)
gdf_nin_vec = gdf_nin_vec[gdf_nin_vec['nin_int'] != NODATA].reset_index(drop=True)
gdf_nin_vec[['NiN_code', 'NiN_name']] = (
    gdf_nin_vec['nin_int'].map(NIN_INT_MAP).apply(pd.Series)
)
gdf_nin_vec


,nin_int,geometry,NiN_code,NiN_name
0,8,"POLYGON ((133900 7052050, 133900 7051950, 1339...",NA-MA06,Afotisk saltvanns-sedimentbunn
1,11,"POLYGON ((134150 7052050, 134150 7051950, 1342...",NA-MA02,Eufotisk fast saltvannsbunn
2,7,"POLYGON ((134200 7052050, 134200 7051950, 1344...",NA-MA05,Eufotisk saltvanns-sedimentbunn
3,12,"POLYGON ((134400 7052050, 134400 7051950, 1345...",NA-MA03,Afotisk fast saltvannsbunn
4,4,"POLYGON ((137750 7052050, 137750 7051950, 1378...",NA-MA01,Fast saltvanns-fjærebeltebunn
...,...,...,...,...
55511,8,"POLYGON ((-7850 6908650, -7850 6908550, -7900 ...",NA-MA06,Afotisk saltvanns-sedimentbunn
55512,11,"POLYGON ((-7700 6908550, -7700 6908450, -7650 ...",NA-MA02,Eufotisk fast saltvannsbunn
55513,11,"POLYGON ((22100 6908550, 22100 6908450, 22150 ...",NA-MA02,Eufotisk fast saltvannsbunn
55514,4,"POLYGON ((22150 6908550, 22150 6908450, 22200 ...",NA-MA01,Fast saltvanns-fjærebeltebunn


## Subtract land (STRtree)

Bulk spatial index finds all (nin_polygon, land_polygon) intersecting pairs at once.
For each NiN polygon, all overlapping land polygons are unioned and subtracted in one call.

In [80]:
def _to_polygons(geom):
    """Extract only polygon parts from any geometry (handles GeometryCollections)."""
    if geom is None or geom.is_empty:
        return geom
    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom
    polys = [
        g for g in getattr(geom, 'geoms', [])
        if isinstance(g, (Polygon, MultiPolygon))
    ]
    if not polys:
        return Polygon()
    return MultiPolygon([p for mp in polys
                         for p in (mp.geoms if isinstance(mp, MultiPolygon) else [mp])])

land = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/aux/Basisdata_Landareal.geo.parquet"
)

nin_geoms = gdf_nin_vec.geometry.values.copy()

tree = STRtree(land.geometry)
nin_idxs, land_idxs = tree.query(nin_geoms, predicate='intersects')

nin_to_land = defaultdict(list)
for n, l in zip(nin_idxs.tolist(), land_idxs.tolist()):
    nin_to_land[n].append(l)

print(f'Subtracting land from {len(nin_to_land):,} of {len(nin_geoms):,} polygons...')
for nin_i, l_indices in nin_to_land.items():
    land_union = unary_union(land.geometry.iloc[l_indices].values)
    nin_geoms[nin_i] = _to_polygons(nin_geoms[nin_i].difference(land_union))

gdf_nin_land = gdf_nin_vec.copy()
gdf_nin_land['geometry'] = nin_geoms
gdf_nin_land = gdf_nin_land[~gdf_nin_land.is_empty].reset_index(drop=True)
gdf_nin_land


Subtracting land from 21,026 of 55,516 polygons...


,nin_int,geometry,NiN_code,NiN_name
0,8,"POLYGON ((133900 7052050, 133900 7051950, 1339...",NA-MA06,Afotisk saltvanns-sedimentbunn
1,11,"POLYGON ((134150 7052050, 134150 7051950, 1342...",NA-MA02,Eufotisk fast saltvannsbunn
2,7,"POLYGON ((134200 7052050, 134200 7051950, 1344...",NA-MA05,Eufotisk saltvanns-sedimentbunn
3,12,"POLYGON ((134400 7052050, 134400 7051950, 1345...",NA-MA03,Afotisk fast saltvannsbunn
4,4,"POLYGON ((137750 7052050, 137750 7051950, 1378...",NA-MA01,Fast saltvanns-fjærebeltebunn
...,...,...,...,...
55511,8,"POLYGON ((-7850 6908650, -7850 6908550, -7900 ...",NA-MA06,Afotisk saltvanns-sedimentbunn
55512,11,"POLYGON ((-7650 6908450, -7700 6908450, -7700 ...",NA-MA02,Eufotisk fast saltvannsbunn
55513,11,"POLYGON ((22100 6908550, 22150 6908550, 22150 ...",NA-MA02,Eufotisk fast saltvannsbunn
55514,4,"POLYGON ((22150 6908550, 22200 6908550, 22200 ...",NA-MA01,Fast saltvanns-fjærebeltebunn


## Save result

In [81]:
fname = subkart.utils.to_filename("nisjemodell", "moere-og-romsdal", "2026", CRS.split(":")[1])
gdf_nin_land.to_file(f'{fname}.gpkg', driver='GPKG', layer='nin')
gdf_nin_land.to_parquet(f'{fname}.geo.parquet', compression='snappy')
print(f'Saved {len(gdf_nin_land):,} polygons → {fname}.gpkg / .geo.parquet')


Saved 55,516 polygons → nisjemodell_moere-og-romsdal_2026_25833.gpkg / .geo.parquet


In [82]:
subkart.utils.to_postgis(gdf_nin_land, fname)

Table nisjemodell_moere_og_romsdal_2026 uploaded to PostGIS.
